In [0]:
from pyspark.sql.functions import col, monotonically_increasing_id
from pyspark.sql.functions import col, explode, sequence, to_date, year, month, dayofmonth, date_format, when

# 1. Load data from the Silver table (The Source of Truth)
# Loading persisted silver data
df_silver_data = spark.table("workspace.silver.cleaned_yellow_taxi")

# 2. Create Dimension: Vendor
df_dim_vendor = df_silver_data.select("VendorID").distinct() \
    .withColumn("vendor_sk", monotonically_increasing_id())
df_dim_vendor.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.dim_vendor")

# 3. Create Dimension: Payment Type
df_dim_payment_type = df_silver_data.select("payment_type").distinct() \
    .withColumn("payment_type_sk", monotonically_increasing_id())
df_dim_payment_type.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.dim_payment_type")


# 4. Create Dimension: Rate Code
df_dim_rate_code = df_silver_data.select("RateCodeID").distinct() \
    .withColumn("rate_code_sk", monotonically_increasing_id())
df_dim_rate_code.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.dim_rate_code")

# 1. Defining the date range for the project
#  Generate a standalone Date Dimension from 2020 to 2026
#  Expanding date dimension to cover data from 2015 onwards
begin_date = '2015-01-01'
end_date = '2026-12-31'

df_dim_date = spark.sql(f"SELECT explode(sequence(to_date('{begin_date}'), to_date('{end_date}'), interval 1 day)) as full_date") \
    .withColumn("date_key", date_format(col("full_date"), "yyyyMMdd").cast("int")) \
    .withColumn("year", year(col("full_date"))) \
    .withColumn("month", month(col("full_date"))) \
    .withColumn("day", dayofmonth(col("full_date"))) \
    .withColumn("day_name", date_format(col("full_date"), "EEEE")) \
    .withColumn("month_name", date_format(col("full_date"), "MMMM")) \
    .withColumn("is_weekend", when(date_format(col("full_date"), "EEEE").isin("Saturday", "Sunday"), 1).otherwise(0))

# Overwriting the table with the new range
df_dim_date.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.gold.dim_date")


# Saving the independent Dimension to Gold layer
df_dim_date.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.dim_date")


print("Dimensions created: dim_date, dim_vendor, dim_payment_type, dim_rate_code")
     